In [ ]:
'''
pip install langchain
pip install dotenv
pip install langchain-community
pip install langchain-google-genai
pip install google-search-results
pip install bs4
pip install requests
'''

In [ ]:
from typing import List, Dict
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from serpapi import GoogleSearch
import os
import re
from dotenv import load_dotenv

In [ ]:
load_dotenv("secrets.env")
serp_key = os.getenv("SERPAPI_API_KEY")

In [229]:
event_categories = ['Business', 'Health', 'Music', 'Charity', 'Social Causes', 'Community','Culture', 'Education', 'Online',
                    'Science', 'Nightlife', 'Food & Drink', 'Outdoors', 'Performance', 'Tastings', 'Food Festival',
                    'Dating', 'LGBTQ', 'Comedy', 'Fitness', 'Tech', 'Home', 'Lifestyle', 'Parties', 'Film', 'Theater']

In [230]:
def search_eventbrite(query: str, location: str, max_results: int = 10) -> List[Dict]:
  
    search_query = f'site:eventbrite.com "{query}" "{location}"'
    
    params = {
        "engine": "google",
        "q": search_query,
        "api_key": serp_key,
        "num": max_results
    }
    
    search = GoogleSearch(params)
    results = search.get_dict()
    
    events = []
    for r in results.get("organic_results", [])[:max_results]:
        events.append({
            "url": r.get("link")
        })
    return events

In [231]:
results = {}

for category in event_categories:
    results[category] = {
        "category": category,
        "events": search_eventbrite(category, "New York", max_results=10)
    }

In [232]:
rows = []
for cat, details in results.items():
    category = details["category"]
    for event in details["events"]:
        rows.append({"category": category, "url": event["url"]})


df = pd.DataFrame(rows)
df["category"] = df["category"].replace("LGBTQ", "Community & Culture")
df["category"] = df["category"].replace("Community", "Community & Culture")
df["category"] = df["category"].replace("Culture", "Community & Culture")
df["category"] = df["category"].replace("Outdoors", "Health & Fitness")
df["category"] = df["category"].replace("Health", "Health & Fitness")
df["category"] = df["category"].replace("Fitness", "Health & Fitness")
df["category"] = df["category"].replace("Home", "Home & Lifestyle")
df["category"] = df["category"].replace("Lifestyle", "Home & Lifestyle")
df["category"] = df["category"].replace("Comedy", "Comedy & Performance")
df["category"] = df["category"].replace("Performance", "Comedy & Performance")
df["category"] = df["category"].replace("Charity", "Charity & Social Causes")
df["category"] = df["category"].replace("Social Causes", "Charity & Social Causes")
df["category"] = df["category"].replace("Science", "STEM")
df["category"] = df["category"].replace("Tech", "STEM")
df["category"] = df["category"].replace("Nightlife", "Nightlife & Parties")
df["category"] = df["category"].replace("Parties", "Nightlife & Parties")
df["category"] = df["category"].replace("Food Festival", "Food & Drink")
df["category"] = df["category"].replace("Tastings", "Food & Drink")
df["category"] = df["category"].replace("Film", "Comedy & Performance")
df["category"] = df["category"].replace("Theater", "Comedy & Performance")
df

,category,url
0,Business,https://www.eventbrite.com/b/ny--new-york/busi...
1,Business,https://www.eventbrite.com/d/ny--new-york/free...
2,Business,https://www.eventbrite.com/d/ny--new-york/busi...
3,Business,https://www.eventbrite.com/d/ny--new-york/busi...
4,Business,https://www.eventbrite.com/d/ny--new-york/busi...
...,...,...
249,Comedy & Performance,https://www.eventbrite.com/b/united-states--ne...
250,Comedy & Performance,https://www.eventbrite.com/o/new-york-public-l...
251,Comedy & Performance,https://www.eventbrite.com/d/ny--new-york/part...
252,Comedy & Performance,https://www.eventbrite.com/b/ny--yonkers/arts/...


In [233]:
df['category'].value_counts()

category
Comedy & Performance       40
Community & Culture        30
Health & Fitness           30
Food & Drink               30
Nightlife & Parties        20
STEM                       20
Home & Lifestyle           19
Charity & Social Causes    15
Business                   10
Music                      10
Online                     10
Education                  10
Dating                     10
Name: count, dtype: int64

In [260]:
def is_actual_event(url: str) -> bool:

    return re.search(r"-tickets-\d+$", url) is not None

def get_event_links(url: str, category: str, idx: int) -> pd.DataFrame:
    try:
        r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        r.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Skipping index {idx}, url={url} due to error: {e}")
        return pd.DataFrame(columns=["category", "url"])  # empty df if failed

    soup = BeautifulSoup(r.text, "html.parser")

    rows = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        rows.append({"category": category, "url": href})

    return pd.DataFrame(rows)


In [ ]:
expanded_rows = []
for idx, row in df.iterrows():
    expanded_rows.append(get_event_links(row["url"], row["category"], idx))

final_df = pd.concat([df] + expanded_rows, ignore_index=True)
final_df['actual'] = final_df['url'].apply(is_actual_event)
final_df = final_df[final_df['actual'] == True]
final_df 

⚠️ Skipping index 162, url=https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669


In [278]:
final_df = final_df[final_df['actual'] == True]
final_df.reset_index()
final_df.drop_duplicates(subset='url', inplace=True)

In [ ]:
# Add a try catch block if website invalid

def get_event_metadata(url: str):
    try:
        session = requests.Session()
        session.headers.update({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 (KHTML, like Gecko) "
                        "Chrome/120.0.0.0 Safari/537.36"
        })

        response = session.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Skipping index url={url} due to error: {e}")
        return pd.DataFrame(columns=["category", "url"])
    

    soup = BeautifulSoup(response.text, "html.parser")


    meta_tags = {meta.get("property") or meta.get("name"): meta.get("content")
                for meta in soup.find_all("meta") if meta.get("content")}

    keys_to_keep = [
        'og:title',
        'og:description',
        'og:url',
        'event:start_time',
        'event:end_time',
        'event:location:latitude',
        'event:location:longitude',  
        'og:image'
    ]   

    filtered_meta_tags = {k: v for k, v in meta_tags.items() if k in keys_to_keep}
    filtered_meta_tags

    return filtered_meta_tags


In [280]:
final_df['metadata'] = final_df['url'].apply(get_event_metadata)

⚠️ Skipping index 253, url=https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/saturday-night-ages-24-36-speed-dating-in-new-york-cityspeed-nyc-tickets-1428426685669
⚠️ Skipping index 253, url=https://www.eventbrite.com/e/moca-talks-with-karen-hao-empire-of-ai-tickets-1602071221149 due to error: 404 Client Error: Not Found for url: https://www.eventbrite.com/e/moca-talks-with-karen-hao-empire-of-ai-tickets-1602071221149


In [ ]:
final_df.reset_index()

actual
True    487
Name: count, dtype: int64

In [287]:
final_df

,category,url,actual,metadata
44,Charity & Social Causes,https://www.eventbrite.com/e/kissed-a-ghoul-li...,True,{'og:image': 'https://cdn.evbuc.com/images/111...
60,Community & Culture,https://www.eventbrite.com/e/sabor-y-cultura-n...,True,{'og:image': 'https://cdn.evbstatic.com/s3-bui...
61,Community & Culture,https://www.eventbrite.com/e/ultimate-new-york...,True,{'og:image': 'https://img.evbuc.com/https%3A%2...
64,Community & Culture,https://www.eventbrite.com/e/nyc-language-cult...,True,{'og:image': 'https://cdn.evbuc.com/images/108...
69,Education,https://www.eventbrite.com/e/2025-education-in...,True,{'og:image': 'https://cdn.evbuc.com/images/104...
...,...,...,...,...
46102,Comedy & Performance,https://www.eventbrite.com/e/movie-night-magma...,True,{'og:image': 'https://cdn.evbuc.com/images/110...
46103,Comedy & Performance,https://www.eventbrite.com/e/film-screening-do...,True,{'og:image': 'https://cdn.evbuc.com/images/113...
46108,Comedy & Performance,https://www.eventbrite.com/e/isabel-coixet-fil...,True,{'og:image': 'https://cdn.evbuc.com/images/110...
46109,Comedy & Performance,https://www.eventbrite.com/e/isabel-coixet-fil...,True,{'og:image': 'https://cdn.evbuc.com/images/110...


In [288]:
final_df.drop(columns=['actual'], inplace=True)

In [292]:
keys = [
    'og:title', 'og:description', 'og:url', 
    'event:start_time', 'event:end_time', 
    'event:location:latitude', 'event:location:longitude', 'og:image'
]

for key in keys:
    final_df[key] = final_df['metadata'].apply(lambda x: x.get(key) if isinstance(x, dict) else None)

final_df

,category,url,metadata,og:title,og:description,og:url,event:start_time,event:end_time,event:location:latitude,event:location:longitude,og:image
44,Charity & Social Causes,https://www.eventbrite.com/e/kissed-a-ghoul-li...,{'og:image': 'https://cdn.evbuc.com/images/111...,"Kissed a Ghoul, Liked It: Diversity in Paranor...",Hear from a panel of authors about writing tra...,https://www.eventbrite.com/e/kissed-a-ghoul-li...,2025-10-26T15:00:00-04:00,2025-10-26T17:00:00-04:00,42.3895917,-71.10617839999999,https://cdn.evbuc.com/images/1117003563/286985...
60,Community & Culture,https://www.eventbrite.com/e/sabor-y-cultura-n...,{'og:image': 'https://cdn.evbstatic.com/s3-bui...,"Sabor Y Cultura NEW YORK Tickets, Jul 12 | Eve...",SABOR is coming to the BIG APPLE!! Join us fo...,https://www.eventbrite.com/e/sabor-y-cultura-n...,None,None,None,None,https://cdn.evbstatic.com/s3-build/perm_001/6c...
61,Community & Culture,https://www.eventbrite.com/e/ultimate-new-york...,{'og:image': 'https://img.evbuc.com/https%3A%2...,Ultimate New York Food & Culture Tour™ in the ...,Ultimate New York Food & Culture Tour™ in the ...,https://www.eventbrite.com/e/ultimate-new-york...,2014-12-08T14:45:00-05:00,2026-03-15T14:30:00-04:00,40.7316603,-74.00314049999997,https://img.evbuc.com/https%3A%2F%2Fcdn.evbuc....
64,Community & Culture,https://www.eventbrite.com/e/nyc-language-cult...,{'og:image': 'https://cdn.evbuc.com/images/108...,NYC Language & Culture Social – Every Friday @...,meet new people from all over the world (and t...,https://www.eventbrite.com/e/nyc-language-cult...,2025-07-25T19:00:00-04:00,2028-09-22T21:00:00-04:00,40.70986449999999,-74.00874859999999,https://cdn.evbuc.com/images/1082246403/283999...
69,Education,https://www.eventbrite.com/e/2025-education-in...,{'og:image': 'https://cdn.evbuc.com/images/104...,2025 Education in New York Summit,Shaping Tomorrow’s Schools Through Innovation,https://www.eventbrite.com/e/2025-education-in...,2025-08-14T09:00:00-04:00,2025-08-14T15:30:00-04:00,40.7059752,-74.01857889999997,https://cdn.evbuc.com/images/1045054723/130664...
...,...,...,...,...,...,...,...,...,...,...,...
46102,Comedy & Performance,https://www.eventbrite.com/e/movie-night-magma...,{'og:image': 'https://cdn.evbuc.com/images/110...,Movie Night - Magma,La Soufrière volcano in Guadeloupe threatens e...,https://www.eventbrite.com/e/movie-night-magma...,2025-10-08T18:30:00-04:00,2025-10-08T20:30:00-04:00,40.7688624,-73.95153169999998,https://cdn.evbuc.com/images/1108061783/321111...
46103,Comedy & Performance,https://www.eventbrite.com/e/film-screening-do...,{'og:image': 'https://cdn.evbuc.com/images/113...,Film Screening - ¡Dolores Guapa!,Film screening followed by a conversation with...,https://www.eventbrite.com/e/film-screening-do...,2025-10-02T18:00:00-04:00,2025-10-02T20:30:00-04:00,40.7308053,-73.9984357,https://cdn.evbuc.com/images/1134323073/277603...
46108,Comedy & Performance,https://www.eventbrite.com/e/isabel-coixet-fil...,{'og:image': 'https://cdn.evbuc.com/images/110...,Isabel Coixet Film Marathon: MY LIFE WITHOUT ME,Join director Isabel Coixet for a screening of...,https://www.eventbrite.com/e/isabel-coixet-fil...,2025-09-27T11:30:00-04:00,2025-09-27T14:00:00-04:00,40.7308053,-73.9984357,https://cdn.evbuc.com/images/1107602273/277603...
46109,Comedy & Performance,https://www.eventbrite.com/e/isabel-coixet-fil...,{'og:image': 'https://cdn.evbuc.com/images/110...,Isabel Coixet Film Marathon: THE SECRET LIFE O...,Join director Isabel Coixet for a screening of...,https://www.eventbrite.com/e/isabel-coixet-fil...,2025-09-27T14:30:00-04:00,2025-09-27T17:00:00-04:00,40.7308053,-73.9984357,https://cdn.evbuc.com/images/1107618263/277603...


In [299]:
final_df = final_df.dropna()

In [300]:
def extract_ticket_id(url):
    match = re.search(r'-tickets-(\d+)', url)
    return match.group(1) if match else None


In [301]:
final_df['event_id'] = final_df['og:url'].apply(extract_ticket_id)

C:\Users\Jason\AppData\Local\Temp\ipykernel_12556\4252023947.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['event_id'] = final_df['og:url'].apply(extract_ticket_id)


In [309]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 446 entries, 44 to 46111
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   category                  446 non-null    object
 1   url                       446 non-null    object
 2   metadata                  446 non-null    object
 3   og:title                  446 non-null    object
 4   og:description            446 non-null    object
 5   og:url                    446 non-null    object
 6   event:start_time          446 non-null    object
 7   event:end_time            446 non-null    object
 8   event:location:latitude   446 non-null    object
 9   event:location:longitude  446 non-null    object
 10  og:image                  446 non-null    object
 11  event_id                  446 non-null    object
dtypes: object(12)
memory usage: 45.3+ KB


In [310]:
final_df.rename(columns={'og:title' : 'title',
                         'og:description' : 'description',
                         'event:start_time' : 'start_time',
                         'event:end_time' : 'end_time',
                         'event:location:latitude' : 'latitude',
                         'event:location:longitude' : 'longitude',
                         'og:image' : 'image'}, inplace=True)

C:\Users\Jason\AppData\Local\Temp\ipykernel_12556\591061100.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns={'og:title' : 'title',


In [312]:
final_df.drop(columns=['url', 'metadata'], inplace=True)

C:\Users\Jason\AppData\Local\Temp\ipykernel_12556\3842691654.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.drop(columns=['url', 'metadata'], inplace=True)


In [318]:
final_df['end_time'] = pd.to_datetime(final_df['end_time'])

C:\Users\Jason\AppData\Local\Temp\ipykernel_12556\992426631.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  final_df['end_time'] = pd.to_datetime(final_df['end_time'])
C:\Users\Jason\AppData\Local\Temp\ipykernel_12556\992426631.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['end_time'] = pd.to_datetime(final_df['end_time'])


In [325]:
final_df= final_df[final_df['end_time'] > pd.Timestamp("2025-09-27 23:59:00-04:00")]
final_df

,category,title,description,og:url,start_time,end_time,latitude,longitude,image,event_id
44,Charity & Social Causes,"Kissed a Ghoul, Liked It: Diversity in Paranor...",Hear from a panel of authors about writing tra...,https://www.eventbrite.com/e/kissed-a-ghoul-li...,2025-10-26T15:00:00-04:00,2025-10-26 17:00:00-04:00,42.3895917,-71.10617839999999,https://cdn.evbuc.com/images/1117003563/286985...,1607191967439
61,Community & Culture,Ultimate New York Food & Culture Tour™ in the ...,Ultimate New York Food & Culture Tour™ in the ...,https://www.eventbrite.com/e/ultimate-new-york...,2014-12-08T14:45:00-05:00,2026-03-15 14:30:00-04:00,40.7316603,-74.00314049999997,https://img.evbuc.com/https%3A%2F%2Fcdn.evbuc....,14814904779
64,Community & Culture,NYC Language & Culture Social – Every Friday @...,meet new people from all over the world (and t...,https://www.eventbrite.com/e/nyc-language-cult...,2025-07-25T19:00:00-04:00,2028-09-22 21:00:00-04:00,40.70986449999999,-74.00874859999999,https://cdn.evbuc.com/images/1082246403/283999...,1507737697039
82,Online,NYC Online Speed Dating (56+ age group),Meet someone special and have fun connecting w...,https://www.eventbrite.com/e/nyc-online-speed-...,2025-05-02T20:00:00-04:00,2025-12-26 22:00:00-05:00,40.7097781,-74.00982710000001,https://cdn.evbuc.com/images/1016449273/273557...,1339454016019
84,Online,NYC Singles Online Trivia Night - A Fun Speed ...,* New York City Online Singles Trivia Night * ...,https://www.eventbrite.com/e/nyc-singles-onlin...,2025-04-01T21:30:00-04:00,2025-10-14 21:00:00-04:00,40.7127753,-74.0059728,https://cdn.evbuc.com/images/975367383/1924315...,1269852475939
...,...,...,...,...,...,...,...,...,...,...
45726,Comedy & Performance,Bushwick Walk-In Studio Portraits,Capture your best angles & personality for our...,https://www.eventbrite.com/e/bushwick-walk-in-...,2025-09-27T10:00:00-04:00,2025-09-28 21:00:00-04:00,40.7066363,-73.9283585,https://cdn.evbuc.com/images/1133247373/288987...,1735584683509
45729,Comedy & Performance,Great Films You May Never See Film Festival,Come check out some awesome films that may hav...,https://www.eventbrite.com/e/great-films-you-m...,2025-09-28T10:00:00-04:00,2025-09-28 21:00:00-04:00,40.813109,-73.9549756,https://cdn.evbuc.com/images/1108238233/332916...,1640765356349
46100,Comedy & Performance,SHORT FILM FESTIVAL - LIVE EVENT - NYC INDIE S...,Get ready for a night of indie short films at ...,https://www.eventbrite.com/e/short-film-festiv...,2025-10-03T13:30:00-04:00,2025-10-05 22:00:00-04:00,40.7714331,-73.993651,https://cdn.evbuc.com/images/1122737033/576770...,1692733715109
46102,Comedy & Performance,Movie Night - Magma,La Soufrière volcano in Guadeloupe threatens e...,https://www.eventbrite.com/e/movie-night-magma...,2025-10-08T18:30:00-04:00,2025-10-08 20:30:00-04:00,40.7688624,-73.95153169999998,https://cdn.evbuc.com/images/1108061783/321111...,1640984311249


In [ ]:
final_df.to_csv('scraped_data.csv', index=False)

In [327]:
final_df['category'].value_counts()

category
Charity & Social Causes    44
Comedy & Performance       44
Health & Fitness           37
Home & Lifestyle           32
Education                  31
STEM                       30
Business                   27
Food & Drink               27
Community & Culture        27
Music                      10
Online                      4
Nightlife & Parties         4
Dating                      1
Name: count, dtype: int64